# Transport Logistics — Dantzig's Classical Transportation Problem

**Continuous Optimization (MasterMath) — Lecture 1 demo.**

The canonical example from G. B. Dantzig, *Linear Programming and Extensions* (1963), Chapter 3-3: two canneries — **Seattle** and **San Diego** — ship cases of a product to three markets — **New York**, **Chicago** and **Topeka**. Freight costs \$90 per case per thousand miles.

| | Supply (cases) | → New York | → Chicago | → Topeka |
|---|---|---|---|---|
| **Seattle** | 350 | 2.5 | 1.7 | 1.8 |
| **San Diego** | 600 | 2.5 | 1.8 | 1.4 |
| **Demand (cases)** | | 325 | 300 | 275 |

(distances in thousands of miles)

**Question.** Which shipping plan meets all demand, within the available supply, at minimal total cost?

With $x_{ij}$ the amount shipped from facility $j$ to consumer $i$, this is the linear program from the lecture:

$$\begin{array}{rl} \min & \sum_{i=1}^n \sum_{j=1}^m c_{ij} x_{ij} \\ \text{s.t.} & x_{ij} \ge 0, \\ & \sum_{i=1}^n x_{ij} \le s_j \quad \text{(supply at facility } j\text{)}, \\ & \sum_{j=1}^m x_{ij} \ge d_i \quad \text{(demand of consumer } i\text{)}. \end{array}$$

In [ ]:
# Colab does not ship CVXPY by default; install it (skipped if already present).
try:
    import cvxpy  # noqa: F401
except ImportError:
    %pip install -q cvxpy

In [ ]:
import numpy as np
import cvxpy as cp

facilities = ["Seattle", "San Diego"]
consumers = ["New York", "Chicago", "Topeka"]

s = np.array([350, 600])       # supply s_j (cases)
d = np.array([325, 300, 275])  # demand d_i (cases)

# Distances in thousands of miles, consumers (rows) x facilities (columns).
dist = np.array([
    [2.5, 2.5],  # New York
    [1.7, 1.8],  # Chicago
    [1.8, 1.4],  # Topeka
])
c = 90 * dist  # marginal cost c_ij in $ per case: freight $90 per case per 1000 miles

# Decision variable: x_ij = amount shipped from facility j to consumer i.
x = cp.Variable((3, 2), nonneg=True)

constraints = [
    cp.sum(x, axis=0) <= s,  # ship no more than each facility stocks
    cp.sum(x, axis=1) >= d,  # meet every consumer's demand
]

objective = cp.Minimize(cp.sum(cp.multiply(c, x)))

problem = cp.Problem(objective, constraints)
problem.solve()

print(f"Status: {problem.status}")
print(f"Minimal total shipping cost = {problem.value:.2f} dollar")
print()
for i, market in enumerate(consumers):
    for j, plant in enumerate(facilities):
        if x.value[i, j] > 1e-6:
            print(f"  {plant:10s} -> {market:9s} : {x.value[i, j]:6.1f} cases")

The minimal cost is the classical answer, **\$153,675**. Note that both canneries are equally far from New York ($c_{ij} = \$225$ per case), so *any* split of the New York demand between them — with the rest of the plan fixed — costs the same: the problem has multiple optimal solutions, and the solver returns one of them. Dantzig's textbook solution ships 50 cases from Seattle and 275 from San Diego.

In [ ]:
# Draw the optimal shipping plan: line width proportional to the amount shipped.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
pos_f = {j: (0, 1 - 2 * j) for j in range(2)}       # facilities on the left
pos_c = {i: (1, 1 - i) for i in range(3)}           # consumers on the right

for i in range(3):
    for j in range(2):
        amount = x.value[i, j]
        if amount > 1e-6:
            (x0, y0), (x1, y1) = pos_f[j], pos_c[i]
            ax.plot([x0, x1], [y0, y1], lw=amount / 30, color="tab:blue", alpha=0.6)
            ax.annotate(f"{amount:.0f}", ((x0 + x1) / 2, (y0 + y1) / 2 + 0.06), ha="center")

for j, name in enumerate(facilities):
    ax.annotate(f"{name}\n(supply {s[j]})", pos_f[j], ha="right", va="center",
                bbox=dict(boxstyle="round", fc="lightyellow"))
for i, name in enumerate(consumers):
    ax.annotate(f"{name}\n(demand {d[i]})", pos_c[i], ha="left", va="center",
                bbox=dict(boxstyle="round", fc="lightgreen"))

ax.set_xlim(-0.35, 1.35)
ax.set_ylim(-2.4, 1.4)
ax.axis("off")
ax.set_title(f"Optimal shipping plan (cases) — total cost ${problem.value:,.0f}")
plt.show()